# **Установка необходимых библиотек**

In [ ]:
!pip install --break-system-packages clickhouse-connect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 11.1 MB/s eta 0:00:0000:010:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 45.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 45.0 MB/s eta 0:00:00


Для занятия:

Файлы лежат в каталоге /var/lib/clickhouse/data/

# **Подключаемся к базе данных**

In [2]:
import clickhouse_connect
import pandas as pd
import os

pd.options.display.max_colwidth = 100

# Вбейте свой телеграм никнейм, чтобы в случае проблем мы могли вас индефицировать
database = 'artemw9'

client = clickhouse_connect.get_client(host='clickhouse01', port=8123, username=os.getenv('CLICKHOUSE_USER'), password=os.getenv('CLICKHOUSE_PASSWORD'))

# **Создаем свое окружение**

In [3]:
client.command(f'''
    CREATE DATABASE IF NOT EXISTS {database} ON CLUSTER '{{cluster}}';
''')

['clickhouse01',
 '9000',
 '0',
 '',
 '3',
 '0\nclickhouse02',
 '9000',
 '0',
 '',
 '2',
 '0\nclickhouse04',
 '9000',
 '0',
 '',
 '1',
 '0\nclickhouse03',
 '9000',
 '0',
 '',
 '0',
 '0']

# **Типы данных**

In [4]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.type_data;
''')

In [5]:
client.command(f'''
    CREATE TABLE {database}.type_data
    (
    --------------------------
    -- основные типы данных --
    --------------------------
        i Int8,                               -- Int8-256 (со знаком)
        ui UInt8,                             -- UInt8-256 (без знака)
        fl Float32,                           -- Float32-64 (для мат.расчетов, но не для финансов)  
        dc Decimal(9, 4),                     -- Decimal32-256 (точность после запятой)
        st String,                            -- имеет произвольную длинну
        fst FixedString(5),                   -- имеет фиксированную длинну
    --------------------------------
    -- дополнитеьлные типы данных --
    --------------------------------
        UID UUID,                             -- уникальный идентификатор
        ip4 IPv4,                             -- 127.0.0.1
        ip6 IPv6,                             -- f2c6:e19b:da60:52ad:2cef:62fe:0279
    --------------------------------
    --    типы даты и времени     --
    --------------------------------
        dt Date,                              -- Date32 (различаются диапозном дат)
        dtm DateTime,                         -- Сохрняет время с точностью то секунд
        dtm64 DateTime64(6),                     -- Сохрняет время с точностью то наносекунд
    ------------------------------------
    --    композитные типы данных             -- позволяют хранить более сложные структуры данных
    ------------------------------------
        ar Array(UInt8),                      -- массив данных
        tu Tuple(Date, UInt16, Decimal32(2)), -- кортеж
        ns Nested(                            -- Вложенные структуры
            col1 String,
            col2 UInt64,
            col3 String
            ),
    mp Map(String, Int16),                    -- хранит в данные в виде ключ -> значение
    en Enum('bad' = 2,                        -- хранит данные определенного значения
            'udovlet' = 3, 
            'good' = 4 )      
    )
    ENGINE = Log;
''')




In [ ]:
client.command(f'''
    INSERT INTO {database}.type_data
    (
        i, ui, fl, dc, st, fst,
        UID, ip4, ip6,
        dt, dtm, dtm64,
        ar, tu, 
        ns.col1, ns.col2, ns.col3,
        mp, en
    )
    VALUES 
    (
        -100,
        200,
        3.14,
        toDecimal32(3.14, 4),
        'Пример строки',
        'ABCDE',
        generateUUIDv4(),
        '192.168.1.1',
        '2001:db8::1',
        toDate('2025-04-30'),
        toDateTime('2025-04-30 14:30:00'),
        toDateTime64('2025-04-30 14:30:00.123456', 6),
        [10, 20, 30],
        (toDate('2025-04-30'), 150, 99.99),
        ['one'],
        [123456],
        ['value1'],
        map('key1', toInt16(10), 'key2', toInt16(-20)),
        'udovlet'
    );
''')

In [6]:
client.query_df(f'''
    SELECT 
        i, ui, fl, dc, st, fst,
        UID, ip4, ip6,
        toString(dt) as dt_str,
        toString(dtm) as dtm_str,
        toString(dtm64) as dtm64_str,
        ar, tu,
        ns.col1, ns.col2, ns.col3,
        mp, en
    FROM {database}.type_data
''')


""


Обращение к композитрым типам данных

In [ ]:
client.query_df(f'''
    SELECT 
        ar[1],      -- обращение к эл-там массива
        ar.size0,   -- получение размера массива
        tu,         -- чтение кортежа
        ns.col1,    -- обращение к вложенной структуре
        mp['key1'], -- получение данных из Map
        en          -- Чтение Enum
    FROM {database}.type_data
''')

,"arrayElement(ar, 1)",ar.size0,tu,ns.col1,"arrayElement(mp, 'key1')",en
0,10,3,"(2025-04-30T00:00:00.000000000, 150, 99.99)",[one],10,udovlet


### Задание на самостоятельную работу


1) Необходимо содать таблицу заказов (Мы еще не проходили создание таблиц поэтому создай по примеру выше с движком `Log`)

| Поле         | Описание                            |
|--------------|-------------------------------------|
| `order_id`   | Уникальный ID заказа                |
| `user_id`    | ID пользователя                     |
| `order_date` | Дата и время оформления заказа      |
| `total_amount` | Сумма заказа                      |
| `paid`       |  Признак оплаты: оплачено, не оплачено|
| `items`      |  Список ID товаров в заказе         |

2) Тебе необходимо выбрать самому подходящие типы данных для колонок
3) После создания вставь подходящие строки через `INSERT INTO <имя БД>.<имя таблицы> VALUES (...)`
4) Выведи выборку строк

In [62]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.orders;
''')

In [63]:
client.command(f'''
    CREATE TABLE {database}.orders
    (
        order_id UInt64,
        user_id UInt64,
        order_date DateTime,
        total_amount Decimal(8, 2),
        paid String,
        items Array(UInt64)
        
    )
    ENGINE = Log;
''')




In [64]:
client.command(f'''
    INSERT INTO {database}.orders
    (
        order_id, user_id,
        order_date, total_amount,
        paid, items
    )
    VALUES 
    (
        1001,  
        12345, 
        now(),
        2999.99, 
        'Yes',
        [101, 102, 103] 
    ),
    (
        1002,
        67890,
        toDateTime('2024-01-15 14:30:00'),
        149.50,
        'No',
        [201, 202]
    ),
    (
        1003,
        55555,
        toDateTime('2024-01-20 09:15:00'),
        5499.99,
        'Yes',
        [301, 302, 303, 304]
    ),
    (
        1004,
        77777,
        toDateTime('2024-01-25 18:45:00'),
        99.99,
        'Pending',
        [401]
    ),
    (
        1005,
        88888,
        toDateTime('2024-01-30 12:00:00'),
        2500.00,
        'Yes',
        [501, 502, 503, 504, 505]
    );
''')

In [66]:
client.query_df(f'''
    SELECT * FROM  {database}.orders
''')

,order_id,user_id,order_date,total_amount,paid,items
0,1001,12345,2026-02-03 15:20:44+03:00,2999.99,Yes,"[101, 102, 103]"
1,1002,67890,2024-01-15 14:30:00+03:00,149.50,No,"[201, 202]"
2,1003,55555,2024-01-20 09:15:00+03:00,5499.99,Yes,"[301, 302, 303, 304]"
3,1004,77777,2024-01-25 18:45:00+03:00,99.99,Pending,[401]
4,1005,88888,2024-01-30 12:00:00+03:00,2500.00,Yes,"[501, 502, 503, 504, 505]"


### Решение(Смотреть только после выполенения)


In [73]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.order;
''')

client.command(f'''
    CREATE TABLE IF NOT EXISTS {database}.order
    (
        order_id     UInt64,
        user_id      UInt32,
        order_date   Date,
        total_amount Decimal(10, 2),
        paid         Enum('оплачено' = 1,
                          'не оплачено' = 0),
        items        Array(UInt32)
    )
    ENGINE = Log
''')

In [74]:
client.command(f'''
    INSERT INTO {database}.order VALUES
        (1, 101, '2025-06-20', 2599.99, 'оплачено', [11, 42, 73]),
        (2, 102, '2025-06-22', 499.50, 'не оплачено', [5, 18]),
        (3, 101, '2025-06-24', 0, 'не оплачено', []);
''')

In [75]:
client.query_df(f'''
    SELECT * FROM  {database}.order
''')

,order_id,user_id,order_date,total_amount,paid,items
0,1,101,2025-06-20,2599.99,оплачено,"[11, 42, 73]"
1,2,102,2025-06-22,499.50,не оплачено,"[5, 18]"
2,3,101,2025-06-24,0.00,не оплачено,[]


In [ ]:
# Enum лучше, чем String в поле paid?

# **Функции к приведению типов данных**

In [67]:
client.query_df('''
    select '1'::Int64
''')

,"CAST('1', 'Int64')"
0,1


In [68]:
# Тут будет ошибка
client.query_df('''
    select NULL::Int64
''')

DatabaseError: Received ClickHouse exception, code: 70, server response: Code: 70. DB::Exception: Cannot convert NULL to a non-nullable type: While processing CAST(NULL, 'Int64'). (CANNOT_CONVERT_TYPE) (version 22.5.4.19 (official build)) (for url http://clickhouse01:8123)

In [69]:
client.query_df('''
    select CAST('1'  as Int8)
''')

,"CAST('1', 'Int8')"
0,1


In [70]:
# Тут будет ошибка
client.query_df('''
   select CAST(NULL as Int8)
''')



DatabaseError: Received ClickHouse exception, code: 70, server response: Code: 70. DB::Exception: Cannot convert NULL to a non-nullable type: While processing CAST(NULL, 'Int8'). (CANNOT_CONVERT_TYPE) (version 22.5.4.19 (official build)) (for url http://clickhouse01:8123)

to<Тип данных><исключение в случае ошибки приведения типа: OrNull, OrZero, OrDefault>

In [71]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.cast_type_data;
''')

client.command(f'''
CREATE TABLE {database}.cast_type_data (
    col String
)
ENGINE = Log;
''')

client.command(f'''
INSERT INTO {database}.cast_type_data values ('1'),('2'),('1a'),('-1')
''')

In [72]:
client.query_df(f'''
    SELECT
        col,
        toInt64OrNull(col),
        toInt8OrZero(col),
        toInt8OrDefault(col, -100),
        toUInt8(-1), toUInt8(-1.1), toUInt8(256) -- выход за пределы преобразует в значение по модулю диапозона
    FROM {database}.cast_type_data
''')

,col,toInt64OrNull(col),toInt8OrZero(col),"toInt8OrDefault(col, -100)",toUInt8(-1),toUInt8(-1.1),toUInt8(256)
0,1,1,1,1,255,255,0
1,2,2,2,2,255,255,0
2,1a,<NA>,0,-100,255,255,0
3,-1,-1,-1,-1,255,255,0


### Задание на самостоятельную работу


1) Выполни запросы в следующей ячейке.
2) Сделай так чтобы UNION ALL выполнился
3) Типы данных, которые не могут быть автоматически преобразованы системой должны быть, как у таблицы 1. В случае коализий преобразования типов данных, необходимо ставить значение NULL.

In [ ]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.table_1;
''')

client.command(f'''
CREATE TABLE  {database}.table_1
(
    order_id     UInt32, 
    user_id      UInt16,  
    total_amount Float32,   
    paid         Nullable(UInt8),     
    item_count   Int8   
)
ENGINE = Log;
''')

client.command(f'''
    INSERT INTO {database}.table_1 VALUES (10001, 321, 1499.50, 1, 3), (10001,  321, 1499.50 , NULL, 3), (10001, 321, , 1, 3);
''')

client.command(f'''
    DROP TABLE IF EXISTS {database}.table_2;
''')

client.command(f'''
CREATE TABLE  {database}.table_2
(
    order_id     Int64, 
    user_id      UInt32,  
    total_amount Float32,   
    paid         UInt8,     
    item_count   String  
)
ENGINE = Log;
''')

client.command(f'''
    INSERT INTO {database}.table_2 VALUES (10001, 123456, 899.99, 0, '5'), (10001, 123456, NULL, 0, '5'), (10001, 123456, 899.99, 0, '5 тут специально вписаны буквы') ;
''')

In [101]:
# Исправлять ошибку
client.query_df(f'''
    SELECT order_id, user_id, toFloat32OrDefault(total_amount,0.::Float32), paid, item_count FROM {database}.table_1
    UNION ALL
    SELECT order_id, user_id, total_amount, paid, toInt8OrNull(item_count) FROM {database}.table_2
''')

,order_id,user_id,"toFloat32OrDefault(total_amount, CAST('0.', 'Float32'))",paid,item_count
0,10001,321,1499.50000,1,3
1,10001,321,1499.50000,<NA>,3
2,10001,321,0.00000,1,3
3,10001,123456,899.98999,0,5
4,10001,123456,0.00000,0,5
5,10001,123456,899.98999,0,<NA>


### Решение(Смотреть только после выполенения)


In [95]:
client.query_df(f'''
    SELECT order_id, user_id, total_amount, paid, item_count 
    FROM {database}.table_1
    UNION ALL
    SELECT order_id, user_id, toFloat32(total_amount), paid, toInt32OrNull(item_count)
    FROM {database}.table_2
''')

,order_id,user_id,total_amount,paid,item_count
0,10001,123456,899.98999,0,5
1,10001,123456,0.00000,0,5
2,10001,123456,899.98999,0,<NA>
3,10001,321,1499.50000,1,3
4,10001,321,1499.50000,<NA>,3
5,10001,321,0.00000,1,3


# **Создание таблиц**

Создание таблиц в ClikHouse очень похоже на создание таблиц в других базах данных, за исключением того что в КХ обязательно нужно указывать движок базы данных. Каждый движок во своему уникален, но самый встречаемый и вообще именно для него и создавался КХ это MergeTree.

In [96]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.mt;
''')

client.command(f'''
    CREATE TABLE IF NOT EXISTS {database}.mt
    (
        id UInt32,
        dt datetime
    )
    ENGINE = MergeTree                                                 -- обязательно нужно указывать движок
    PRIMARY KEY(id)                                                    -- не обязательное поле по умолчанию равно ORDER BY
    ORDER BY(id, dt)                                                   -- обязательно должны быть колонки в порядке из primary key
    TTL dt + INTERVAL 1 MONTH DELETE;                                  -- от времени dt отсчитывается 1 месяц после чего данные удаляются (данные удаляются полсе слияния)
        --dt + INTERVAL 1 MONTH DELETE WHERE toDayOfWeek(d) IN (6, 7); -- удаление с фильтрацией
        --dt + INTERVAL 1 WEEK TO VOLUME 'aaa',                        -- перемещение данных в вольюм(совокупность дисков, задаваемая в конфигах)
        --dt + INTERVAL 2 WEEK TO DISK 'bbb';                          -- перемещение данных на диск (указывается имя диска)
''')



### Самостоятельная работа на проверку работоспостобности TTL:

1) Создай таблицу с движком MergeTree c 2мя колонками: id Int32, dt DateTime
2) Укажи только сотировку по id. PRIMARY KEY указывать не нужно оно будет совпадать по умолчанию с ORDER BY
3) Напиши в условии TTL удаление строки по dt через 10 секунд
4) вставь данные через конструкцию `INSERT INTO <имя БД>.<имя таблицы> AS SELECT number, now() + number FROM numbers(100)` (`numbers(100)` это таблица которая генерирует колонку number со значениями от 0 до 99)
5) выполни select твоей таблицы и посмотри сколько в ней строк
6) как я уже говорил удаление происходит после слияния поэтому выполни команду ` OPTIMIZE TABLE <имя БД>.<имя таблицы> FINAL;` через 20 секунд после выполнения шага 5
7) выполни повторно шаг 5
8) теперь вопроряй 5-6 шаги пока в таблице совсем не останется данных

In [4]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.mt;
''')

client.command(f'''
    CREATE TABLE IF NOT EXISTS {database}.mt
    (
        id Int32,
        dt datetime
    )
    ENGINE = MergeTree                                                 -- обязательно нужно указывать движок
    ORDER BY(id)                                                   -- обязательно должны быть колонки в порядке из primary key
    TTL dt + INTERVAL 10 SECOND DELETE;                                  -- от времени dt отсчитывается 1 месяц после чего данные удаляются (данные удаляются полсе слияния)
''')


In [7]:
client.command(f'''
INSERT INTO {database}.mt  SELECT number , now() + number  FROM numbers(100)''')

In [9]:
client.command(f'''
   SELECT * FROM system.parts WHERE table = '{database}.mt'; ''')

In [17]:
client.command(f'''
    OPTIMIZE TABLE {database}.mt FINAL; ''')

In [18]:
client.command(f'''
    SELECT COUNT(*) FROM {database}.mt ''')

52

### Решение(Смотреть только после выполенения)

In [ ]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.table_ttl;
''')

client.command(f'''
    CREATE TABLE IF NOT EXISTS {database}.table_ttl
    (
        id Int32,
        dt DateTime
    )
    ENGINE = MergeTree 
    ORDER BY (id)
    TTL dt + INTERVAL 10 SECOND DELETE;
''')

In [ ]:
client.command(f'''
    INSERT INTO {database}.mt
    SELECT 
      number,
      now() + number
    from numbers(100);
''')

In [ ]:
client.command(f'''
    OPTIMIZE TABLE {database}.mt FINAL;
''')

# **Описание полей при создании таблиц**


In [19]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.test_fields_without_ttl;
''')


client.command(f'''
    CREATE TABLE {database}.test_fields_without_ttl
    (
          col_default UInt64 DEFAULT 42                         -- значение по умолчанию
        , col_materialized UInt64 MATERIALIZED col_default * 33 -- к данной колонке можно обратиться только по имени
        , col_alias UInt64 ALIAS col_default + 1                -- к данной колонке можно обратиться только по имени
        , col_codec String CODEC(ZSTD(10))                      -- кодек сжатия
        , col_comment Date COMMENT 'Some comment'               -- комментарий к колонке
    )
    ENGINE = Log;
''')

In [20]:
client.command(f'''
    INSERT INTO {database}.test_fields_without_ttl (
        col_default,
        col_codec,
        col_comment
    )
    SELECT
        number,
        'какой-то текст ' || toString(number),
        toDate(now()) + number
    FROM numbers(60);
''')

In [21]:
client.query_df(f'''
    SELECT * FROM {database}.test_fields_without_ttl
''')

,col_default,col_codec,col_comment
0,0,какой-то текст 0,2026-02-04
1,1,какой-то текст 1,2026-02-05
2,2,какой-то текст 2,2026-02-06
3,3,какой-то текст 3,2026-02-07
4,4,какой-то текст 4,2026-02-08
5,5,какой-то текст 5,2026-02-09
6,6,какой-то текст 6,2026-02-10
7,7,какой-то текст 7,2026-02-11
8,8,какой-то текст 8,2026-02-12
9,9,какой-то текст 9,2026-02-13


А теперь посмотрим как работает TTL для колонкок

In [25]:


client.command(f'''
    DROP TABLE IF EXISTS {database}.test_fields_with_ttl;
''')


client.command(f'''
    CREATE TABLE {database}.test_fields_with_ttl
    (
          col_default UInt64 DEFAULT 42
        , col_materialized UInt64 MATERIALIZED col_default * 33
        , col_alias UInt64 ALIAS col_default + 1
        , col_codec String CODEC(ZSTD(10))
        , col_comment Date COMMENT 'Some comment'
        , col_ttl UInt64 DEFAULT 10  TTL col_comment + INTERVAL 5 DAY
    )
    ENGINE = MergeTree()
    ORDER BY (col_default);
''')

client.command(f'''
    INSERT INTO {database}.test_fields_with_ttl (
        col_default,
        col_codec,
        col_comment,
        col_ttl
    )
        SELECT
            number,
            'какой-то текст' ||  toString(number),
            toDate(now()) - number,
            rand(1) % 100000000
        FROM numbers(20);
''')

In [26]:
client.query_df(f'''
    SELECT 
        col_default
      , col_materialized 
      , col_alias 
      , col_codec 
      , col_comment
      , col_ttl
    FROM {database}.test_fields_with_ttl
''')

,col_default,col_materialized,col_alias,col_codec,col_comment,col_ttl
0,0,0,1,какой-то текст0,2026-02-04,77181301
1,1,33,2,какой-то текст1,2026-02-03,10355914
2,2,66,3,какой-то текст2,2026-02-02,90917912
3,3,99,4,какой-то текст3,2026-02-01,94102690
4,4,132,5,какой-то текст4,2026-01-31,46285868
5,5,165,6,какой-то текст5,2026-01-30,10
6,6,198,7,какой-то текст6,2026-01-29,10
7,7,231,8,какой-то текст7,2026-01-28,10
8,8,264,9,какой-то текст8,2026-01-27,10
9,9,297,10,какой-то текст9,2026-01-26,10


С помощью следующей команды можно увидеть комментарий к столбцу

In [31]:
client.query_df(f'''
    DESCRIBE TABLE {database}.test_fields_with_ttl -- здесь можно увидеть комментарий к столбцу
''')


,name,type,default_type,default_expression,comment,codec_expression,ttl_expression
0,col_default,UInt64,DEFAULT,42,,,
1,col_materialized,UInt64,MATERIALIZED,col_default * 33,,,
2,col_alias,UInt64,ALIAS,col_default + 1,,,
3,col_codec,String,,,,ZSTD(10),
4,col_comment,Date,,,Some comment,,
5,col_ttl,UInt64,DEFAULT,10,,,col_comment + toIntervalDay(5)


### Самостоятельная работа на расшиненные атрибуты колонок:

1) Создайте таблицу `product_sales`, которая будет хранить информацию о продажах товаров. В таблице необходимо использовать **все указанные выше атрибуты хотя бы по одному разу**.

| Поле          | Тип данных  | Атрибуты                                                              |
|---------------|-------------|------------------------------------------------------------------------|
| `sale_id`     | `UInt64`    | `COMMENT`: "Уникальный идентификатор продажи"                        |
| `price`       | `Float32`   | `DEFAULT`: 0.0                                                        |
| `quantity`    | `UInt8`     | `DEFAULT`: 1                                                          |
| `total`       | `Float32`   | `MATERIALIZED`: `price * quantity`                                   |
| `description` | `String`    | `CODEC`: `ZSTD(1)`                                                    |
| `taxed_total` | `Float32`   | `ALIAS`: `total * 1.2`
2) Вставь пару строк в данную таблицу данные в данную таблицу
3) Выведи данные через колонки и через `*`.
4) Попробуй вставить данные в колонки `total` и `taxed_total`. Посмотри на результат

In [ ]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.test_fields_with_b;
''')


client.command(f'''
    CREATE TABLE {database}.test_fields_with_b
    (
          sale_id UInt64 COMMENT 'Some comment'
        , price Float32 DEFAULT 0.0
        , quantity UInt8 DEFAULT 1
        , total Float32 MATERIALIZED price * quantity
        , description String CODEC(ZSTD(1))
        , taxed_total Float32 ALIAS total * 1.2
    )
    ENGINE = MergeTree()
    ORDER BY (sale_id);
''')

client.command(f'''
    INSERT INTO {database}.test_fields_with_b (
        sale_id,
        price,
        quantity,
        description
    )
        SELECT
            number,
            number * 10,
            (number + 10) - number * 0.5,
            'какой-то текст' ||  toString(number)
        FROM numbers(20);
''')



In [ ]:
client.query_df(f'''
    SELECT 
        sale_id
      , price 
      , quantity 
      , total 
      , description
      , taxed_total
    FROM {database}.test_fields_with_b
''')

,sale_id,price,quantity,description
0,0,0.0,10,какой-то текст0
1,1,10.0,10,какой-то текст1
2,2,20.0,11,какой-то текст2
3,3,30.0,11,какой-то текст3
4,4,40.0,12,какой-то текст4
5,5,50.0,12,какой-то текст5
6,6,60.0,13,какой-то текст6
7,7,70.0,13,какой-то текст7
8,8,80.0,14,какой-то текст8
9,9,90.0,14,какой-то текст9


In [ ]:
client.command(f'''
    INSERT INTO {database}.test_fields_with_b (

        taxed_total
    )
        SELECT
            number,
            number * 10
        FROM numbers(10);
''')

DatabaseError: Received ClickHouse exception, code: 16, server response: Code: 16. DB::Exception: No such column taxed_total in table artemw9.test_fields_with_balbes (8ade7804-14a6-434e-801f-9153d994bca9). (NO_SUCH_COLUMN_IN_TABLE) (version 22.5.4.19 (official build)) (for url http://clickhouse01:8123)

### Решение(Смотреть только после выполенения)

In [11]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.product_sales;
''')


client.command(f'''
    CREATE TABLE {database}.product_sales
    (
        sale_id UInt64 COMMENT 'Уникальный идентификатор продажи',
        price Float32 DEFAULT 0.0,
        quantity UInt8 DEFAULT 1,
        total Float32 MATERIALIZED price * quantity,
        description String CODEC(ZSTD(1)),
        taxed_total Float32 ALIAS total * 1.2
    )
    ENGINE = MergeTree
    ORDER BY sale_id;
''')

client.command(f'''
    INSERT INTO {database}.product_sales (sale_id, price, quantity, description) VALUES
        (1, 10.5, 2, 'Покупка товара А'),
        (2, 5.0, 5, 'Покупка товара Б');
''')

In [12]:
client.query_df(f'''
    SELECT 
        sale_id
      , price 
      , quantity 
      , total 
      , description
      , taxed_total
    FROM {database}.product_sales
''')

,sale_id,price,quantity,total,description,taxed_total
0,1,10.5,2,21.0,Покупка товара А,25.200001
1,2,5.0,5,25.0,Покупка товара Б,30.000000


Сравни результаты со следующим запросом. Как ты можешь увидеть не хватает 2х колонок

In [59]:
client.query_df(f'''
    SELECT 
        *
    FROM {database}.product_sales
''')

,sale_id,price,quantity,description
0,1,10.5,2,Покупка товара А
1,2,5.0,5,Покупка товара Б


Вставлять данные в материализованные и алиас колонки нельзя. Поверь в следующем запросе

In [60]:
client.command(f'''
    INSERT INTO {database}.product_sales (sale_id, price, quantity, total, taxed_total, description) VALUES
    (3, 7.0, 3, 21.0, 25.2, 'Покупка товара В');
''')
# При выполнении произойдет ОШИБКА!!
# Cannot insert column total, because it is MATERIALIZED column.


DatabaseError: Received ClickHouse exception, code: 44, server response: Code: 44. DB::Exception: Cannot insert column total, because it is MATERIALIZED column. (ILLEGAL_COLUMN) (version 22.5.4.19 (official build)) (for url http://clickhouse01:8123)

# **Атрибуты при создании колонок**

In [120]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.nl_lc_tabl;
''')

client.command(f'''
    CREATE TABLE {database}.nl_lc_tabl (
        a Nullable(UInt32),         -- разрешает вставку с пропуском значения.
        b LowCardinality(String),   -- ускоряет работу малокоординальных данных(часто повторяющихся)
        c UInt32
    ) ENGINE = MergeTree 
    ORDER BY tuple(); -- определяет порядок строк по порядку вставки данных
''')

client.command(f'''
    INSERT INTO {database}.nl_lc_tabl VALUES (NULL,'test' ,1); -- null вставится корректно
''')
client.command(f'''
    INSERT INTO {database}.nl_lc_tabl VALUES (1,'test2',NULL); -- null вставится как 0.
''')
client.command(f'''
    INSERT INTO {database}.nl_lc_tabl VALUES (1, NULL, 3); -- null вставляется как пустая строка
''')
client.command(f'''
    INSERT INTO {database}.nl_lc_tabl VALUES (1, 'test2', 4);
''')

In [182]:
client.query_df(f'''
    SELECT 
        a, 
        b, 
        c 
    FROM {database}.nl_lc_tabl
''')

# А зачем рандомный порядок?

,a,b,c
0,1,,3
1,1,test2,4
2,1,test2,0
3,<NA>,test,1


Сущесвует парамерт который запретит вставлять данные NULL в не Nullable колонки. При которой будут ошибки

In [122]:

client.command(f'''
    SET input_format_null_as_default = 0; -- параметр отвечающий за вставку пустых значений. Выполняется совместно с командой на вставку
''')
# появится ошибка при вставке
client.command(f'''
    INSERT INTO {database}.nl_lc_tabl VALUES (2, 'sdasda', NULL)
''')


DatabaseError: Received ClickHouse exception, code: 53, server response: Code: 53. DB::Exception: Cannot insert NULL value into a column of type 'UInt32' at: NULL)
: While executing ValuesBlockInputFormat. (TYPE_MISMATCH) (version 22.5.4.19 (official build)) (for url http://clickhouse01:8123)

In [123]:
client.query_df(f'''
    SELECT 
        toTypeName(a), 
        toTypeName(b), 
        toTypeName(c) 
    FROM {database}.nl_lc_tabl
''')

,toTypeName(a),toTypeName(b),toTypeName(c)
0,Nullable(UInt32),LowCardinality(String),UInt32
1,Nullable(UInt32),LowCardinality(String),UInt32
2,Nullable(UInt32),LowCardinality(String),UInt32
3,Nullable(UInt32),LowCardinality(String),UInt32


# **Партицирование**

**Партицирование** - это процесс объединения наборов данных по единому критерию, например по месяцу. Это позволяет ускорить процес считывания данных по выбранному критерию при фильтрации.
В ClickHouse существует 4 типа партицирования

## Диапозоном

In [129]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.table_range;
''')

client.command(f'''
    CREATE TABLE {database}.table_range
    (
        id UInt32,
        name String,
        created_at Date
    )
    ENGINE = MergeTree
    PARTITION BY
        CASE
            WHEN id < 10000 THEN 'range_1'
            WHEN id < 20000 THEN 'range_2'
            ELSE 'range_3'
        END
    ORDER BY id;
''')

client.command(f'''
    INSERT INTO {database}.table_range
    SELECT
        number AS id,
        concat('User_', toString(number)) AS name,
        today() AS created_at
    FROM
        numbers(30000)
''')

client.command(f'''
    OPTIMIZE TABLE {database}.table_range FINAL;
''')

In [130]:
# посмотреть какие у таблицы партиции
client.query_df(f'''
    select 
        _part,
        count(*) 
    FROM 
        {database}.table_range 
    GROUP BY _part 
''')

,_part,count()
0,784ee64cfdda054ca6737d6f17bcf9a9_2_2_1,10000
1,d9cb78d099336586dcc3052fcabd64d1_3_3_1,10000
2,a5b5a22b2816f91588be1cd95ffed316_1_1_1,10000


## Интервалом

Чаще всего в проектах, даже не больших, встречается именно этот вид партицирования

In [3]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.table_interval;
''')

client.command(f'''
    CREATE TABLE {database}.table_interval
    (
        id UInt32,
        amount Float32,
        sale_date Date
    )
    ENGINE = MergeTree
    PARTITION BY toYYYYMM(sale_date)
    ORDER BY id;
''')

client.command(f'''
    INSERT INTO {database}.table_interval
    SELECT
        number AS id,
        rand() % 1000 AS amount,
        today() + (number % 90) AS sale_date
    FROM numbers(1000);
''')

client.command(f'''
    OPTIMIZE TABLE {database}.table_interval FINAL;
''')

In [6]:
# посмотреть какие у таблицы партиции
client.query_df(f'''
    select
  
        _part,
        count(*) 
    FROM 
        {database}.table_interval 
    GROUP BY _part 
''')

,_part,count()
0,202604_3_3_1,330
1,202605_4_4_1,88
2,202602_1_1_1,241
3,202603_2_2_1,341


## хеш-функцией

In [7]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.table_hash;
''')

client.command(f'''
    CREATE TABLE {database}.table_hash
    (
        user_id UInt64,
        event String
    )
    ENGINE = MergeTree
    PARTITION BY cityHash64(user_id) % 10
    ORDER BY user_id;
''')

client.command(f'''
    INSERT INTO {database}.table_hash
    SELECT
        number AS user_id,
        concat('event_', toString(number))
    FROM numbers(1000);
''')

client.command(f'''
    OPTIMIZE TABLE {database}.table_hash FINAL;
''')

In [8]:
# посмотреть какие у таблицы партиции
client.query_df(f'''
    select 
        _part,
        count(*) 
    FROM 
        {database}.table_hash 
    GROUP BY _part 
''')

,_part,count()
0,8_5_5_1,94
1,4_4_4_1,89
2,7_6_6_1,94
3,5_8_8_1,116
4,0_1_1_1,109
5,1_7_7_1,109
6,3_9_9_1,102
7,9_3_3_1,87
8,2_2_2_1,88
9,6_10_10_1,112


## комбинированое 

In [ ]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.table_composiste;
''')

client.command(f'''
    CREATE TABLE {database}.table_composiste
    (
        order_id UInt64,
        customer_id UInt64,
        order_date Date
    )
    ENGINE = MergeTree
    PARTITION BY (toYYYYMM(order_date), customer_id % 10)
    ORDER BY order_id;
''')

client.command(f'''
    INSERT INTO {database}.table_composiste
    SELECT
        number AS order_id,
        number % 100 AS customer_id,
        today() + (number % 90) AS order_date
    FROM numbers(1000);
''')

client.command(f'''
    OPTIMIZE TABLE {database}.table_composiste FINAL;
''')



In [10]:
# посмотреть какие у таблицы партиции
client.query_df(f'''
    select 
        _part,
        count(*) 
    FROM 
        {database}.table_composiste 
    GROUP BY _part 
''')

,_part,count()
0,202602-9_10_10_1,23
1,202605-4_33_33_1,11
2,202604-1_30_30_1,33
3,202602-1_2_2_1,23
4,202605-7_36_36_1,11
5,202604-9_28_28_1,33
6,202604-2_21_21_1,33
7,202603-1_11_11_1,44
8,202604-5_24_24_1,33
9,202602-6_7_7_1,23


### Самостоятельная работа на партиционирование таблиц:
1) Создайте таблицу `customer_orders` с партиционированием по месяцу `PARTITION BY toYYYYMM(order_date)` и первичным ключом сортирвки  `ORDER BY` - `customer_id, order_date`. Поле `total` должно быть `MATERIALIZED`. 

| Поле             | Тип         | Описание                                 |
|------------------|-------------|------------------------------------------|
| `order_id`       | UInt64      | Уникальный идентификатор заказа          |
| `order_date`     | DateTime    | Дата и время заказа                      |
| `customer_id`    | UInt32      | ID клиента                               |
| `product_name`   | String      | Название товара                          |
| `quantity`       | UInt8       | Кол-во единиц                            |
| `price`          | Float32     | Цена за единицу                          |
| `total`          | Float32     | MATERIALIZED: `quantity * price`         |

2) Вставьте данные за несколько разных месяцев по  (можно использовать `now() - INTERVAL X DAY`)
4) Выведи данные на экран с помощью `*` и убедись в какой парции нахаодятся твои данные добавив скрытую колонку `_part`

In [ ]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.customer_orders;
''')
client.command(f'''
    CREATE TABLE {database}.customer_orders
    (
        order_id UInt64,
        order_date DateTime,
        customer_id UInt32,
        product_name String,
        quantity UInt8,
        price Float32,
        total Float32 MATERIALIZED quantity * price
    )
    ENGINE = MergeTree
    PARTITION BY toYYYYMM(order_date)
    ORDER BY (customer_id, order_date);
''')

client.command(f'''
    INSERT  INTO {database}.customer_orders
    (order_id, order_date, customer_id, product_name, quantity, price)
    VALUES
        (1001, '2024-03-15 10:30:00', 501, 'Laptop Pro', 1, 1299.99),
        (1002, '2024-03-15 10:30:00', 501, 'Wireless Mouse', 2, 29.99),
        (1003, '2024-03-14 14:45:00', 501, 'USB-C Cable', 3, 19.99),
        (1004, '2024-03-14 14:45:00', 417, 'Keyboard', 1, 89.99),
        (1005, '2024-03-13 12:30:00', 302, 'Monitor 24"', 1, 249.99),
        (1006, '2024-03-13 12:30:00', 638, 'Laptop Pro', 1, 1299.99),
        (1007, '2024-04-13 12:30:00', 501, 'Mouse Pad', 5, 9.99),
        (1008, '2024-04-10 17:00:00', 417, 'Webcam HD', 2, 49.99),
        (1009, '2024-04-10 17:00:00', 417, 'Headphones', 1, 79.99),
        (1010, '2024-04-10 17:00:00', 417, 'USB-C Cable', 4, 19.99);
''')



client.command(f'''
    OPTIMIZE TABLE {database}.customer_orders FINAL;
''')




DatabaseError: Received ClickHouse exception, code: 62, server response: Code: 62. DB::Exception: Syntax error (Multi-statements are not allowed): failed at position 78 (end of query) (line 4, col 18): ;

 FORMAT Native. . (SYNTAX_ERROR) (version 22.5.4.19 (official build)) (for url http://clickhouse01:8123)

In [21]:
client.query_df(f'''
    SELECT _part, * FROM {database}.customer_orders 

''')

,_part,order_id,order_date,customer_id,product_name,quantity,price
0,202404_2_2_1,1008,2024-04-10 17:00:00+03:00,417,Webcam HD,2,49.990002
1,202404_2_2_1,1009,2024-04-10 17:00:00+03:00,417,Headphones,1,79.989998
2,202404_2_2_1,1010,2024-04-10 17:00:00+03:00,417,USB-C Cable,4,19.990000
3,202404_2_2_1,1007,2024-04-13 12:30:00+03:00,501,Mouse Pad,5,9.990000
4,202403_1_1_1,1005,2024-03-13 12:30:00+03:00,302,"Monitor 24""",1,249.990005
5,202403_1_1_1,1004,2024-03-14 14:45:00+03:00,417,Keyboard,1,89.989998
6,202403_1_1_1,1003,2024-03-14 14:45:00+03:00,501,USB-C Cable,3,19.990000
7,202403_1_1_1,1001,2024-03-15 10:30:00+03:00,501,Laptop Pro,1,1299.989990
8,202403_1_1_1,1002,2024-03-15 10:30:00+03:00,501,Wireless Mouse,2,29.990000
9,202403_1_1_1,1006,2024-03-13 12:30:00+03:00,638,Laptop Pro,1,1299.989990


### Решение(Смотреть только после выполенения)

In [17]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.customer_orders;
''')

client.command(f'''
    CREATE TABLE {database}.customer_orders
    (
        order_id UInt64,
        order_date DateTime,
        customer_id UInt32,
        product_name String,
        quantity UInt8,
        price Float32,
        total Float32 MATERIALIZED quantity * price
    )
    ENGINE = MergeTree
    PARTITION BY toYYYYMM(order_date)
    ORDER BY (customer_id);
''')

client.command(f'''
    INSERT INTO {database}.customer_orders (order_id, order_date, customer_id, product_name, quantity, price) VALUES
        (1, now() - INTERVAL 100 DAY, 101, 'Товар A', 2, 150.0),
        (2, now() - INTERVAL 65 DAY, 102, 'Товар B', 1, 300.0),
        (3, now() - INTERVAL 32 DAY, 103, 'Товар C', 3, 200.0),
        (4, now(), 101, 'Товар A', 5, 120.0);
''')

In [18]:
client.query_df(f'''
    SELECT _part, * FROM {database}.customer_orders 
    WHERE order_date  >= '2025-04-01'
''')

,_part,order_id,order_date,customer_id,product_name,quantity,price
0,202510_1_1_0,1,2025-10-31 14:22:24+03:00,101,Товар A,2,150.0
1,202512_2_2_0,2,2025-12-05 14:22:24+03:00,102,Товар B,1,300.0
2,202601_3_3_0,3,2026-01-07 14:22:24+03:00,103,Товар C,3,200.0
3,202602_4_4_0,4,2026-02-08 14:22:24+03:00,101,Товар A,5,120.0


# **Движки таблиц**

**SummingMergeTree** - таблица с группировкой одинаковых записей по ключу сортировки и применением суммы к перечисленным полям 

In [22]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.summing_mt;
''')

client.command(f'''
    CREATE TABLE {database}.summing_mt
    (
        id UInt32,
        val UInt32,
        dt datetime,
        example UInt32  -- столбец, не входящий в ключ сортировки и параметры движка
    )
    ENGINE = SummingMergeTree(val) -- сумма будет считаться по полю val, так как оно указано в качестве параметра движка 
    ORDER BY (id)
    PARTITION BY toYYYYMM(dt); -- записи по этому ключу будут группироваться
''')

client.command(f'''
    INSERT INTO {database}.summing_mt
    SELECT 
        number % 2, 
        (number + 1) * 10, 
        now() + number * 60 * 60 * 24,
        (number + 1) * 100 
    from numbers(30);
''')

Запомни что данные сливаются только в рамках партиции

In [23]:
client.query_df(f'''
    select * from {database}.summing_mt
''')

,id,val,dt,example
0,0,1040,2026-03-02 14:34:59+03:00,2300
1,1,1300,2026-03-01 14:34:59+03:00,2200
2,0,1210,2026-02-08 14:34:59+03:00,100
3,1,1100,2026-02-09 14:34:59+03:00,200


Давай добавим еще данных и  посмотрим на разницу с предыдущим запросом

In [24]:
client.command(f'''
    INSERT INTO {database}.summing_mt
    SELECT 
        number % 2, 
        (number + 1) * 10, 
        now() + number * 60 * 60 * 24,
        (number + 1) * 100 
    from numbers(30);
''')

In [25]:
client.query_df(f'''
    select * from {database}.summing_mt
''')

,id,val,dt,example
0,0,1040,2026-03-02 14:35:45+03:00,2300
1,1,1300,2026-03-01 14:35:45+03:00,2200
2,0,1040,2026-03-02 14:34:59+03:00,2300
3,1,1300,2026-03-01 14:34:59+03:00,2200
4,0,1210,2026-02-08 14:34:59+03:00,100
5,1,1100,2026-02-09 14:34:59+03:00,200
6,0,1210,2026-02-08 14:35:45+03:00,100
7,1,1100,2026-02-09 14:35:45+03:00,200


Как мы видим создался новый блок данных, теперь эти блоки необходими слить друг с другом

In [26]:
client.command(f'''
    OPTIMIZE TABLE {database}.summing_mt FINAL; -- ручное слияние
''')

In [27]:
client.query_df(f'''
    select * from {database}.summing_mt
''')

,id,val,dt,example
0,0,2420,2026-02-08 14:34:59+03:00,100
1,1,2200,2026-02-09 14:34:59+03:00,200
2,0,2080,2026-03-02 14:34:59+03:00,2300
3,1,2600,2026-03-01 14:34:59+03:00,2200


Чтобы увидеть конечный результат и в рамках ключа сортировки оставалось одно значение, то необходимо добавить ключевое слово FINAL в конструкцию FROM. Обрати внимание что данные просуммировались

In [28]:
client.query_df(f'''
    select * from {database}.summing_mt FINAL
''')

,id,val,dt,example
0,0,4500,2026-02-08 14:34:59+03:00,100
1,1,4800,2026-02-09 14:34:59+03:00,200


**AggregatingMergeTree** -- это таблица, которая группирует одинаковые записи по ключу сортировки и применяет агрегатные функции к полям

#### Комбинаторы агрегатных функций
https://clickhouse.com/docs/ru/sql-reference/aggregate-functions/combinators

In [ ]:
client.query_df('''
    with t1 as (
    select number, 
            number * 10 as colsum,
            number % 3 as coldist
    from numbers(10)
    )
    select 
        sumIf(colsum, number % 2 == 0),
        countDistinct(coldist),
        countDistinctIf(coldist, coldist % 2 = 0)
    from t1
''')

,"sumIf(colsum, equals(modulo(number, 2), 0))",uniqExact(coldist),"countDistinctIf(coldist, equals(modulo(coldist, 2), 0))"
0,200,3,2


### Самостоятельная работа на комбинаторы:

1) Выполни запрос запрос, чтобы создать тестовую таблицу
2) Напиши 3 запроса отвечающие на следующие вопросы:
- Сколько уникальных пользователей заходили с **мобильных устройств**
- Среднее время просмотра только на странице `home`
- Для каждого устройства — количество уникальных пользователей
- Собери список страниц, на которые заходил пользователь в индефикатором `2`, только с десктопа (`web`)

**Подсказака**:
В запросах нужно исполльзовать следующие компненты: `groupArray`, `If`, `avg`, `uniq`

In [31]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.page_views;
''')

client.command(f'''
    CREATE TABLE {database}.page_views
    (
        user_id UInt32,
        page String,
        device String,
        view_time UInt8
    )
    ENGINE = MergeTree
    ORDER BY user_id;
''')

client.command(f'''
    INSERT INTO {database}.page_views (user_id, page, device, view_time) VALUES
        (1, 'home', 'mobile', 10),
        (1, 'catalog', 'mobile', 15),
        (2, 'home', 'web', 8),
        (2, 'checkout', 'web', 20),
        (2, 'home', 'mobile', 5),
        (4, 'catalog', 'web', 12),
        (5, 'checkout', 'mobile', 7),
        (6, 'home', 'web', 9),
        (7, 'home', 'mobile', 11),
        (8, 'catalog', 'web', 13);
''')

In [59]:
client.query_df('''
    select 
        countDistinctIf(user_id, device == 'mobile') as first,
        avgIf(view_time, page = 'home') as second,

        groupArrayIf([page], device == 'web' and user_id == 2) as fourth
        from artemw9.page_views
''')

,first,second,fourth
0,4,8.6,"[[home], [checkout]]"


In [58]:
client.query_df('''
    select 
        device,
        countDistinct(user_id) as third
        from artemw9.page_views
        Group by device
''')

,device,third
0,web,4
1,mobile,4


### Решение(Смотреть только после выполенения)

2.1 Сколько уникальных пользователей заходили с **мобильных устройств**

In [45]:
client.query_df(f'''
    SELECT 
        uniqIf(user_id, device = 'mobile') AS mobile_users 
    FROM {database}.page_views
''')

,mobile_users
0,4


2.2 Среднее время просмотра только на странице `home`

In [46]:
client.query_df(f'''
    SELECT 
        avgIf(view_time, page = 'home') AS avg_home_time 
    FROM {database}.page_views
''')

,avg_home_time
0,8.6


2.3 Для каждого устройства — количество уникальных пользователей

In [48]:
client.query_df(f'''
    SELECT 
        device, 
        uniq(user_id) 
    FROM {database}.page_views
    GROUP BY device
''')

,device,uniq(user_id)
0,web,4
1,mobile,4


2.4 Собери список страниц, на которые заходил пользователь в индефикатором `2`, только с десктопа (`web`)

In [49]:
client.query_df(f'''
    SELECT
        user_id,
        groupArrayIf(page, device = 'web') AS web_pages_visited
    FROM {database}.page_views
    WHERE user_id = 2
    GROUP BY user_id    
''')

,user_id,web_pages_visited
0,2,"[home, checkout]"


#### Агрегаторные типы данных

Сущесвует 2 агригаторных типа данных:
- SimpleAggregateFunction - предназначет для хранения простых агрегатов, которое хранит конечно состояние
- AggregateFunction - сложные агрегаты, которая хранит состояние всех добавленных значений

Разберем пример с **SimpleAggregateFunction**

In [7]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.simple;
''')

client.command(f'''
    CREATE TABLE {database}.simple (
      id UInt64, 
      val_sum SimpleAggregateFunction(sum, UInt64), -- предусмотрен для хранения простых агрегатов(хранит конечное состояние)
      val_max SimpleAggregateFunction(max, UInt32)
    ) 
    ENGINE=AggregatingMergeTree 
    ORDER BY id;
''')

client.command(f'''
    INSERT INTO {database}.simple SELECT  1, number, number from numbers(10);
''')
client.command(f'''
    INSERT INTO {database}.simple SELECT  2, sum(number), max(number) from numbers(5);
''')
client.command(f'''
    INSERT INTO {database}.simple SELECT  1, number, number from numbers(8);
''')

Выведим запрос и увидем, что движок агрегирует данные в блоке данных при вставке. В итоге мы увидем 3 строки

In [8]:
client.query_df(f'''
    SELECT * FROM {database}.simple
''')

,id,val_sum,val_max
0,2,10,4
1,1,45,9
2,1,28,7


Но нам же нужно для ID=1 получить 1 строку. Можно дождаться следующего слияния, а можно выполнитьследующий запрос, которое сделает **логическое** слияние и выдаст конечный результат

In [ ]:
client.query_df(f'''
    SELECT * FROM {database}.simple FINAL
''')

,id,val_sum,val_max
0,2,10,4
1,1,73,9


На больших таблицах намного эффективнее получить тот же результат следущим запросом

In [10]:
# лучше делать так

client.query_df(f'''
    SELECT 
        id, 
        sum(val_sum),
        max(val_max) 
    FROM {database}.simple
    GROUP BY id
''')

,id,sum(val_sum),max(val_max)
0,2,10,4
1,1,73,9


Разберем пример с **AggregateFunction**

Здесь немного все посложнее так как нужно использовать специальные комбинаторы

Комбинаторы агрегаторных типов данных:
* SimpleState — возвращает результат агрегирующей функции типа SimpleAggregateFunction.
* State — возвращает промежуточное состояние типа AggregateFunction, используется при вставке.
* Merge — берёт множество состояний, объединяет их и возвращает результат полной агрегации данных.
* MergeState — выполняет слияние промежуточных состояний агрегации, возвращает промежуточное состояние агрегации.

In [19]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.aggr_func_tbl;
''')

client.command(f'''
    CREATE TABLE {database}.aggr_func_tbl
    (
        id UInt64,
        val_uniq AggregateFunction(uniq, UInt64),         -- Хранит в себе промежуточное состояние данных
        val_any AggregateFunction(anyIf, String, UInt8),
        val_quant AggregateFunction(quantiles(0.5, 0.9), UInt64)
    ) ENGINE=AggregatingMergeTree 
    ORDER BY id;
''')

client.command(f'''
    INSERT INTO {database}.aggr_func_tbl
    SELECT 
        1, 
        uniqState(toUInt64(rnd)),                 -- кол-во уникальных значений
        anyIfState(toString(rnd),rnd%2=0),
        quantilesState(0.5, 0.9)(toUInt64(rnd)) 
    FROM
        (SELECT arrayJoin(arrayMap(i -> i * 10, range(10))) as rnd);
''')

In [20]:
# вставь эту строку в бобра иначе не выполнится
client.query_df(f'''
    SELECT 
    id,
    hex(val_uniq) as val_uniq_hex,
    hex(val_any) as val_any_hex, 
    hex(val_quant) as val_quant_hex
    FROM {database}.aggr_func_tbl; 
''')

DatabaseError: Received ClickHouse exception, code: 62, server response: Code: 62. DB::Exception: Syntax error (Multi-statements are not allowed): failed at position 157 (end of query) (line 7, col 31): ; 

 FORMAT Native. . (SYNTAX_ERROR) (version 22.5.4.19 (official build)) (for url http://clickhouse01:8123)

In [21]:
client.query_df(f'''
       SELECT uniqMerge(val_uniq), 
              quantilesMerge(0.5, 0.9)(val_quant), 
              anyIfMerge(val_any) 
       FROM {database}.aggr_func_tbl
''')
 

,uniqMerge(val_uniq),"quantilesMerge(0.5, 0.9)(val_quant)",anyIfMerge(val_any)
0,10,"[45.0, 81.0]",0


In [24]:
client.query_df(f'''
    SELECT * FROM {database}.aggr_func_tbl;
''')
 

DatabaseError: Received ClickHouse exception, code: 62, server response: Code: 62. DB::Exception: Syntax error (Multi-statements are not allowed): failed at position 41 (end of query) (line 2, col 40): ;

 FORMAT Native. . (SYNTAX_ERROR) (version 22.5.4.19 (official build)) (for url http://clickhouse01:8123)

Рассмотрим еще один пример. Обращаю внимание на нахождение среднего.

In [25]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.aggregating_table;
''')

client.command(f'''
    CREATE TABLE {database}.aggregating_table
    (
        id UInt32,
        val_count AggregateFunction(count, UInt64),
        val_avg   AggregateFunction(avg, Float64),
        val_groupArray AggregateFunction(groupArray, UInt64)
    )
    ENGINE = AggregatingMergeTree
    ORDER BY (id);
''')

client.command(f'''
    INSERT INTO {database}.aggregating_table
    with t1 as (
    SELECT number % 4 as id,
        (number + 1) * 1 col1,
        (number + 1) * 1 col2,
        (number + 1) * 1 col3
    from numbers(10)
    )
    select
        id, 
        countState(col1),
        avgState(toFloat64(col2)), -- Обратите внимание на toFloat64. ClickHouse не может автоматически привести 
                                    -- avgState(UInt64) → avgState(Float64), даже если кажется, 
                                    -- что avg всё равно возвращает float.
        groupArrayState(col3)
    from t1
    group by id; 
''')

In [30]:
# напоминаю что такой результат выдаст ошибку только так FORMAT Vertical

client.query_df(f'''
    SELECT  id, hex(val_count) hex_count, hex(val_avg) hex_avg, hex(val_groupArray) hex_group  FROM {database}.aggregating_table;
''')

DatabaseError: Received ClickHouse exception, code: 62, server response: Code: 62. DB::Exception: Syntax error (Multi-statements are not allowed): failed at position 127 (end of query) (line 2, col 126): ;

 FORMAT Native. . (SYNTAX_ERROR) (version 22.5.4.19 (official build)) (for url http://clickhouse01:8123)

In [ ]:
client.query_df(f'''
    SELECT
        id,
        countMerge(val_count)        AS count_val,
        avgMerge(val_avg)            AS avg_val,
        groupArrayMerge(val_groupArray) AS grouped_vals
    FROM {database}.aggregating_table
    group by id
''')

# получается, что вы сохранили сначала avg(10, 20, 30), а затем avg(1, 2, 3). 
# Итоговый результат, который вы получите, будет avg(1, 2, 3, 10, 20, 30). 

### Самостоятельная работа на агригационные движки:

1) Давай возьмем запро, который мы рассматривали при изучени SummingMergeTree(он в следующей ячеке)
2) Измени его таким образом чтобы агрегировалась **Сумма** по `val` и **максимальное заначение** по дате.
3) Вравни значения движка SummingMergeTree и AggregatingMergeTree

In [31]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.summing_mt_dz;
''')

client.command(f'''
    CREATE TABLE {database}.summing_mt_dz
    (
        id UInt32,
        val UInt32,
        dt datetime,
        example UInt32  -- столбец, не входящий в ключ сортировки и параметры движка
    )
    ENGINE = SummingMergeTree(val) -- сумма будет считаться по полю val, так как оно указано в качестве параметра движка 
    ORDER BY (id)
    PARTITION BY toYYYYMM(dt); -- записи по этому ключу будут группироваться
''')

client.command(f'''
    INSERT INTO {database}.summing_mt_dz
    SELECT 
        number % 2, 
        (number + 1) * 10, 
        now() + number * 60 * 60 * 24,
        (number + 1) * 100 
    from numbers(30);
''')

In [32]:
client.query_df(f'''
    SELECT * FROM {database}.summing_mt_dz
''')

,id,val,dt,example
0,0,1210,2026-02-08 23:47:47+03:00,100
1,1,1100,2026-02-09 23:47:47+03:00,200
2,0,1040,2026-03-02 23:47:47+03:00,2300
3,1,1300,2026-03-01 23:47:47+03:00,2200


In [36]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.summing_mt_dz;
''')

client.command(f'''
    CREATE TABLE {database}.summing_mt_dz
    (
        id UInt32,
        val_sum AggregateFunction(sum, UInt64),         -- Хранит в себе промежуточное состояние данных
        val_max AggregateFunction(amx UInt8),
        example UInt32  -- столбец, не входящий в ключ сортировки и параметры движка
    )
    ENGINE = AggregatingMergeTree -- сумма будет считаться по полю val, так как оно указано в качестве параметра движка 
    ORDER BY (id)
    PARTITION BY toYYYYMM(dt); -- записи по этому ключу будут группироваться
''')

client.command(f'''
    INSERT INTO {database}.summing_mt_dz
    SELECT 
        number % 2, 
        (number + 1) * 10, 
        now() + number * 60 * 60 * 24,
        (number + 1) * 100 
    from numbers(30);
''')

DatabaseError: Received ClickHouse exception, code: 36, server response: Code: 36. DB::Exception: Unexpected AST element passed as aggregate function name for data type AggregateFunction. Must be identifier or function. (BAD_ARGUMENTS) (version 22.5.4.19 (official build)) (for url http://clickhouse01:8123)

In [35]:
client.command(f'''
    SELECT 
        id, 
        sum(val),
        max(dt) 
    FROM {database}.simple
    GROUP BY id
''')

DatabaseError: Received ClickHouse exception, code: 47, server response: Code: 47. DB::Exception: Missing columns: 'dt' 'val' while processing query: 'SELECT id, sum(val), max(dt) FROM artemw9.simple GROUP BY id', required columns: 'id' 'val' 'dt', maybe you meant: ['id']. (UNKNOWN_IDENTIFIER) (version 22.5.4.19 (official build)) (for url http://clickhouse01:8123)

### Решение(Смотреть только после выполенения)

In [ ]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.summing_mt_dz;
''')

client.command(f'''
    CREATE TABLE {database}.summing_mt_dz
    (
        id UInt32,
        val SimpleAggregateFunction(sum, UInt64),
        dt SimpleAggregateFunction(max, DateTime),
        example UInt32 
    )
    ENGINE=AggregatingMergeTree 
    ORDER BY id
    PARTITION BY toYYYYMM(dt);
''')

client.command(f'''
    INSERT INTO {database}.summing_mt_dz
    SELECT 
        number % 2, 
        (number + 1) * 10, 
        now() + number * 60 * 60 * 24,
        (number + 1) * 100 
    from numbers(30);
''')

In [ ]:
client.query_df(f'''
    SELECT * FROM {database}.summing_mt_dz
''')

 **ReplacingMergeTree** -- удаляет дублирующиеся записи с одинаковым значением ключа сортировки.

In [3]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.replacing_merge_tree;
''')

client.command(f'''
    CREATE TABLE {database}.replacing_merge_tree
    (
        id UInt32,
        dt date,
        val UInt32
    )
    ENGINE = ReplacingMergeTree(id)
    ORDER BY (id, dt);
''')

In [10]:
client.command(f'''
    INSERT INTO {database}.replacing_merge_tree
    SELECT 1, 
        now()::date,
        (number + 1) * 400
    FROM numbers(1); 
''')

In [13]:
client.query_df(f'''
    SELECT * FROM {database}.replacing_merge_tree
''')

,id,dt,val
0,1,2026-02-09,400


In [12]:
client.command(f'''
    OPTIMIZE TABLE {database}.replacing_merge_tree;
''')

In [7]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.replacing_merge_tree_with_version;
''')

client.command(f'''
    CREATE TABLE {database}.replacing_merge_tree_with_version
    (
        id UInt32,
        dt date,
        val UInt32
    )
    ENGINE = ReplacingMergeTree(dt) -- dt может быть и числовой колонкой
    ORDER BY (id)
    PARTITION BY toYYYYMM(dt);
''')

In [14]:
client.command(f'''
    INSERT INTO {database}.replacing_merge_tree_with_version
    SELECT 
        1, 
        now()::date + number - 15,
        (number + 1) * 1000
    FROM numbers(10);
''')

In [17]:
client.query_df(f'''
    SELECT * FROM {database}.replacing_merge_tree_with_version
''')


,id,dt,val
0,1,2026-01-31,7000
1,1,2026-02-03,10000


In [16]:
client.command(f'''
    OPTIMIZE TABLE {database}.replacing_merge_tree_with_version FINAL;
''')

**CollapsingMergeTree** -- Удаляет дубликаты по ключу сортировки в зависимости от флага

In [32]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.Books;
''')

client.command(f'''
    CREATE TABLE {database}.Books
    (
        ID UInt64,
        Page UInt8,
        Sign Int8 -- имеет только 2 значения "1" и "-1"
    )
    ENGINE = CollapsingMergeTree(Sign)
    ORDER BY ID;
''')

In [33]:
client.command(f'''
    INSERT INTO {database}.Books values (1, 1, 1);
''')

In [35]:
client.query_df(f'''
    SELECT * FROM {database}.Books
''')

,ID,Page,Sign
0,1,1,-1
1,1,2,1
2,1,1,1


In [34]:
client.command(f'''
    INSERT INTO {database}.Books values (1, 1, -1),(1, 2, 1);
''')

In [38]:
client.query_df(f'''
    SELECT * FROM {database}.Books
''')

,ID,Page,Sign
0,1,2,1


In [37]:

client.command(f'''
   OPTIMIZE TABLE {database}.Books;
''')

# в рамках ключа будет оставаться всегда последняя добавленая строка с "+1". 
# все строки с "-1" будут удалены 

**Log** -- для небольших таблиц. Каждая колока отдельный файл

In [40]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.el;
''')

client.command(f'''
    CREATE TABLE {database}.el
    (
        id UInt32,
        dt date
    )
    ENGINE = Log
''')

client.command(f'''
    INSERT INTO {database}.el
    select 
    number,
    now()::date + number
    from numbers(10);
''')

In [41]:
client.query_df(f'''
    SELECT * FROM {database}.el
''')

,id,dt
0,0,2026-02-09
1,1,2026-02-10
2,2,2026-02-11
3,3,2026-02-12
4,4,2026-02-13
5,5,2026-02-14
6,6,2026-02-15
7,7,2026-02-16
8,8,2026-02-17
9,9,2026-02-18


**File** -- позволяет записывать данные в формате файла

In [42]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.ef;
''')

client.command(f'''
    CREATE TABLE {database}.ef
    (
        id UInt32,
        dt date
    )
    ENGINE = File(CSV);
''')

client.command(f'''
    INSERT INTO {database}.ef
    SELECT 
      number,
      now()::date + number
    FROM numbers(10);
''')

In [43]:
client.query_df(f'''
    SELECT * FROM {database}.ef
''')

,id,dt
0,0,2026-02-09
1,1,2026-02-10
2,2,2026-02-11
3,3,2026-02-12
4,4,2026-02-13
5,5,2026-02-14
6,6,2026-02-15
7,7,2026-02-16
8,8,2026-02-17
9,9,2026-02-18


**Buffer** -- для укорения вставки в таблицы. Данные записываются в ОП далее сливаются в другую таблицу

In [52]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.eb;
''')

client.command(f'''
    DROP TABLE IF EXISTS {database}.ebt;
''')

client.command(f'''
    CREATE TABLE {database}.eb
    (
        id UInt32,
        dt date
    )
    ENGINE = Buffer({database}, -- имя БД
                    ebt,     -- имя таблицы для слива данных
                    16,      -- параллелизм (рекомендация 16)
                    30,      -- минимальное время слияния
                    60,      -- минимальное время слияния
                    5,       -- минимальное кол-во строк для слияния
                    10,      -- максимальное кол-во строк для слияния
                    10000,   -- минимальное кол-во байт для слияния
                    10000    -- максимальное кол-во байт для слияния
                    );
''')



In [53]:
client.command(f'''
    INSERT INTO {database}.eb
    select 
        number,
        now()::date + number
    from numbers(1);
''')

In [55]:
# будет ошибка
client.query_df(f'''
    SELECT * FROM {database}.eb
''')

,id,dt
0,0,2026-02-09


In [54]:
client.command(f'''
    CREATE TABLE {database}.ebt
        (
        id UInt32,
        dt date
    )
    ENGINE = Log;
''')

In [58]:
client.query_df(f'''
    SELECT * FROM {database}.ebt
''')

,id,dt
0,0,2026-02-09


In [57]:
client.command(f'''
    OPTIMIZE TABLE {database}.eb;
''')

**Memory** --данные хранятся только в оперативной памяти. При перезапуске CH данные будут утеряны

In [59]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.em;
''')

client.command(f'''
    CREATE TABLE {database}.em
    (
        id UInt32,
        dt date
    )
    ENGINE = Memory;
''')

client.command(f'''
    INSERT INTO {database}.em
    SELECT 
        number,
        now()::date + number
    FROM numbers(100);
''')





In [60]:
client.query_df(f'''
    SELECT * FROM {database}.em
''')

,id,dt
0,0,2026-02-09
1,1,2026-02-10
2,2,2026-02-11
3,3,2026-02-12
4,4,2026-02-13
...,...,...
95,95,2026-05-15
96,96,2026-05-16
97,97,2026-05-17
98,98,2026-05-18


**Set** -- Движок предназначен для использования в правой части оператора IN. Не хранятся дублирующие значения

In [69]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.es;
''')

client.command(f'''
    CREATE TABLE {database}.es
    (
        id UInt32
    )
    ENGINE = Set
    SETTINGS persistent = 1; -- данные будут считываться из ОП
''')

client.command(f'''
    INSERT INTO {database}.es SELECT number from numbers(30);
''')

In [62]:
# читать данные из такой таблицы нельзя, только IN только хардкор
client.query_df(f'''
    SELECT * FROM {database}.es
''')

DatabaseError: Received ClickHouse exception, code: 48, server response: Code: 48. DB::Exception: Method read is not supported by storage Set. (NOT_IMPLEMENTED) (version 22.5.4.19 (official build)) (for url http://clickhouse01:8123)

In [70]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.est;
''')

client.command(f'''
    CREATE TABLE {database}.est
    (
        id UInt32
    )
    ENGINE = MergeTree
    ORDER BY (id);
''')

client.command(f'''
    INSERT INTO {database}.est SELECT number from numbers(300);
''')

In [71]:
client.query_df(f'''
    SELECT *
    FROM {database}.est
    WHERE id in {database}.es
''')

,id
0,0
1,1
2,2
3,3
4,4
5,5
6,6
7,7
8,8
9,9


In [72]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.es2;
''')

client.command(f'''
    DROP TABLE IF EXISTS {database}.est2;
''')

client.command(f'''
    CREATE TABLE {database}.es2
    (
        id UInt32,
        id2 UInt32
    )
    ENGINE = Set
    SETTINGS persistent = 1;
''')

client.command(f'''
    CREATE TABLE {database}.est2
    (
        id UInt32,
        id2 UInt32
    )
    ENGINE = MergeTree
    ORDER BY (id);
''')

client.command(f'''
    INSERT INTO {database}.es2 SELECT number, number from numbers(30);
''')

client.command(f'''
    INSERT INTO {database}.est2 SELECT number, number from numbers(300);
''')

In [73]:
client.query_df(f'''
    SELECT *
    FROM {database}.est2
    WHERE (id, id2) in {database}.es2
''')

,id,id2
0,0,0
1,1,1
2,2,2
3,3,3
4,4,4
5,5,5
6,6,6
7,7,7
8,8,8
9,9,9


**GenerateRandom** -- предназначен для генерации данных в СН для дальнейших тестов

In [74]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.eg;
''')

client.command(f'''
    CREATE TABLE {database}.eg
    (
        id UInt32, 
        val String,
        dt date,
        a Float32,
        b UUID,
        c Bool,
        d IPv6,
        e IPv4,
        g Array(UInt32)
    )
    ENGINE = GenerateRandom;
''')

In [75]:
client.query_df(f'''
    select * 
    from {database}.eg
    limit 10
''')

,id,val,dt,a,b,c,d,e,g
0,3450941728,,2111-03-20,1.116606e+24,810bcf18-d7c4-0ae3-6300-f242e6da022e,True,97c2:a621:cb2b:28c5:7f51:248e:df34:6e16,213.196.93.106,[4221184578]
1,170644377,(Y >^rt2YN,2092-12-26,2.190394e-17,caffbaad-73b4-b75d-60da-ce5579dab0f5,True,4810:bd46:b077:7115:257d:bd76:ae95:364f,71.104.233.218,[]
2,1659363788,)<a/Q8|IC,2010-11-08,-9.808357e+23,a3aab027-eef2-b109-2d30-0262ed858608,True,57bd:a92a:6e9f:a37:4fdc:adce:29ca:9e5f,239.158.223.210,"[1149685657, 429596180, 1260111427, 3367308475, 1200482684, 1157244083, 1298526452]"
3,790338289,yx}DrW,2020-10-06,7.473392e+37,d2b7630c-78f0-9e74-b27c-ba69930e5d60,True,116:63dc:8d03:dff1:3e9c:8bd5:dc85:2ec3,5.246.62.231,"[2755476191, 755913760]"
4,3510702130,!go,2008-08-31,7.180038e+02,e889e6e1-16f7-af60-378b-88a260e05ede,True,766b:e909:ec74:47d7:7690:28b:b30a:447d,94.231.42.84,[]
5,3296833336,,2121-12-26,-1.288091e-05,a24e58e6-1aa9-4da1-2028-a02ea635e6c2,True,df32:15ed:a64a:9316:20c0:c0af:62de:6b78,102.98.65.178,"[1169488199, 2894958324, 913484799, 86796113, 1460492593]"
6,4163971025,"*gzFj""",2138-04-07,-3.298134e+25,13e4bba8-c479-c2c0-2728-eb432f653a98,True,a17a:c391:3c6b:5496:1a79:ea1a:c3ae:2960,10.243.38.155,"[3161338488, 930960147, 1768014515, 3897727025, 2185825505, 3345704011, 540941630, 3528642124, 3..."
7,4219746794,@GFbv;,2101-11-28,6.164684e+10,e75bb3dc-3d16-48f7-da8d-e309a15b21b9,True,e7b4:6e9c:b8a4:52fb:6acf:a839:196c:ae9a,156.38.182.192,"[3255722163, 3926110788, 530911573, 2167907692, 2196442547, 851145624, 652361765, 56743923]"
8,1831865131,Qx,2073-02-16,-1.509564e+38,37c4d404-72d4-dc3c-dc66-031c4670724a,True,18cd:50ad:dcf7:4de4:f7bc:77:6ef:7147,63.162.121.8,[1787751003]
9,1288626515,6Zv,2058-06-02,-1.509265e-03,c34c1f1c-c7be-90e8-e6dd-76b5a32b53f2,True,dcf6:5549:d328:e98:d549:a10a:6e92:6177,12.194.138.56,"[2867988914, 4057685231, 1725528850, 3516925587]"


**PostgreSQL** -- для работы с таблицами БД PSQL

In [ ]:
import psycopg2 as ps
import pandas as pd
import os


schema = 'artemw9' # В расках схемы задайте свою фамилию

conn = ps.connect(host="postgres_source", 
                  port = 5432, 
                  database="dev", 
                  user=os.getenv("POSTGRES_USER"), 
                  password=os.getenv("POSTGRES_PASSWORD"))

cursor = conn.cursor()

cursor.execute(f'''
    CREATE SCHEMA IF NOT EXISTS {schema};
    ''')
    
cursor.execute(f'''
    DROP TABLE IF EXISTS {schema}.departments CASCADE;
''')

cursor.execute(f'''
    CREATE TABLE {schema}.departments (
        dept_id SERIAL PRIMARY KEY,
        dept_name VARCHAR(50),
        location VARCHAR(50)
    )
''')

cursor.execute(f'''
    INSERT INTO {schema}.departments (dept_name, location) VALUES
    ('HR', 'Москва'),
    ('IT', 'Санкт-Петербург'),
    ('Finance', 'Москва'),
    ('DE', 'Краснодар')
''')

conn.commit()

In [84]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.postgresql_table;
''')

client.command(f'''
    CREATE TABLE {database}.postgresql_table
    (
        dept_id Int32,
        dept_name String,
        location String
    )
    ENGINE = PostgreSQL(
                        'postgres_source:5432', -- сервер, порт
                        'dev',              -- БД 
                        'departments',      -- таблица
                        '{os.getenv("POSTGRES_USER")}',         -- логин
                        '{os.getenv("POSTGRES_PASSWORD")}',         -- пароль
                        'artemw9'         -- имя схемы
                        );
''')

In [85]:
client.query_df(f'''
     select * from {database}.postgresql_table
''')

,dept_id,dept_name,location
0,1,HR,Москва
1,2,IT,Санкт-Петербург
2,3,Finance,Москва
3,4,DE,Краснодар


### Самостоятельная работа на движок PSQL

1) Выполни подключение к любой своей таблице и выполни селект запрос

In [106]:
import psycopg2 as ps
import pandas as pd
import os

schema = 'artemw9' # В расках схемы задайте свою фамилию

conn = ps.connect(host="postgres_source", 
                  port = 5432, 
                  database="dev", 
                  user=os.getenv("POSTGRES_USER"), 
                  password=os.getenv("POSTGRES_PASSWORD"))

cursor = conn.cursor()

cursor.execute(f'''
    CREATE SCHEMA IF NOT EXISTS {schema};
    ''')
    
cursor.execute(f'''
    DROP TABLE IF EXISTS {schema}.cars CASCADE;
''')

cursor.execute(f'''
    CREATE TABLE {schema}.cars (
        car_id SERIAL PRIMARY KEY,
        car_vin VARCHAR(50),
        cost INT
    )
''')

cursor.execute(f'''
    INSERT INTO {schema}.cars (car_vin, cost) VALUES
    ('1HGCM82633A123456', 2500000),  
    ('JTDKB20U987654321', 1800000),
    ('WDDGF8BB7AF123456', 3500000),
    ('ZFF67ALA5F0123456', 1200000);
''')

conn.commit()




In [108]:
client.command(f'''
    DROP TABLE IF EXISTS {schema}.postgre_table;
''')

client.command(f'''
    CREATE TABLE {schema}.postgre_table
    (
        car_id Int32,
        car_vin VARCHAR(50),
        cost Int32
    )
    ENGINE = PostgreSQL(
                        'postgres_source:5432', -- сервер, порт
                        'dev',              -- БД 
                        'cars',      -- таблица
                        '{os.getenv("POSTGRES_USER")}',         -- логин
                        '{os.getenv("POSTGRES_PASSWORD")}',         -- пароль
                        'artemw9'         -- имя схемы
                        );
    ''')

client.query_df(f'''SELECT * FROM {schema}.postgre_table ''')

,car_id,car_vin,cost
0,1,1HGCM82633A123456,2500000
1,2,JTDKB20U987654321,1800000
2,3,WDDGF8BB7AF123456,3500000
3,4,ZFF67ALA5F0123456,1200000


**Kafka** - с помощью данного движка можно, как **отправлять** данные в кафку так и **получать** данные из кафки

**Отправляем** данные в кафку

In [10]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.kafka_out_message;
''')

client.command(f'''
    CREATE TABLE {database}.kafka_out_message
    (
        id UInt32
    )
    ENGINE = Kafka
    SETTINGS
        kafka_broker_list = 'kafka:29093',
        kafka_topic_list = '{database}.clickhouse.topic', -- создай такой же топик
        kafka_group_name = 'clickhouse_consumer_group',
        kafka_format = 'JSONEachRow';
''')   

In [11]:
client.command(f'''
    INSERT INTO {database}.kafka_out_message
    SELECT number FROM numbers(30);
''')

После вставки данных проверь свой топик на сообщения

**Читаем** данные из кафки

In [ ]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.kafka_input_stage;
''')

client.command(f'''
    CREATE TABLE {database}.kafka_input_stage
    (
        json_kafka String                               -- задается только 1 строка с полным JSON-ом из кафки
    )
    ENGINE = Kafka
    SETTINGS
        kafka_broker_list = 'kafka:29093',
        kafka_topic_list = 'source.public.order_events',
        kafka_group_name = 'clickhouse_consumer_group',
        kafka_format = 'JSONAsString';                  -- достаточно часто истользуется именно этот формат
''')

Давайте посмотрим, храниться в самой таблице

In [30]:
# Читать из такой таблицы нельзя, при выполенении такого запроса будет ошибка
client.query_df(f'''
     SELECT * FROM {database}.kafka_input_stage
''')

DatabaseError: Received ClickHouse exception, code: 620, server response: Code: 620. DB::Exception: Direct select is not allowed. To enable use setting `stream_like_engine_allow_direct_select`. (QUERY_NOT_ALLOWED) (version 22.5.4.19 (official build)) (for url http://clickhouse01:8123)

Следующим способом можно посмотреть приходят ли данные или нет

In [6]:
client.command(f'''
    SET stream_like_engine_allow_direct_select = 1
''')

client.query_df(f'''
     SELECT * FROM {database}.kafka_input_stage
''')

""


Теперь посмотрим как правильно хранить данные в ClickHouse

In [7]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.table_stage_from_kafka;
''')

client.command(f'''
   create table {database}.table_stage_from_kafka 
   (
      json_kafka String
   )
   Engine = Log
''')

client.command(f'''
    DROP VIEW IF EXISTS {database}.kafka_input_stage_mv;
''')

client.command(f'''
   CREATE MATERIALIZED VIEW {database}.kafka_input_stage_mv TO {database}.table_stage_from_kafka AS 
      SELECT
         json_kafka
      FROM {database}.kafka_input_stage;
''')

Как прочитать данные из строки в формате JSON

In [8]:
client.query_df(f'''
     SELECT  
          JSONExtractInt(JSONExtractString(json_kafka ,'before'), 'id') as before_id,               -- JSONExtract<тип данных>(<JSON строка>, <ключ>)
          JSONExtractInt(JSONExtractString(json_kafka ,'before'), 'order_id') as before_order_id,
          JSONExtractString(JSONExtractString(json_kafka ,'before'), 'status') as before_status,
          JSONExtractUInt(JSONExtractString(json_kafka ,'before'), 'ts') as before_ts,
          JSONExtractInt(JSONExtractString(json_kafka ,'after'), 'id') as after_id,
          JSONExtractInt(JSONExtractString(json_kafka ,'after'), 'order_id') as after_order_id,
          JSONExtractString(JSONExtractString(json_kafka ,'after'), 'status') as after_status,
          JSONExtractUInt(JSONExtractString(json_kafka ,'after'), 'ts') as after_ts,
          JSONExtractString(json_kafka ,'op') as op,
          toDateTime(JSONExtractInt(json_kafka ,'ts_ms') / 1000) as dt                              -- Перевод времени из timestamp в читаемый вид
     FROM {database}.table_stage_from_kafka
''')



""


### Самостоятельная работа на движок Kafka

Ты уже много чего прошел, пришло время давать серьезные задачи, как из реально дают в компаниях.(Это реальный кейс с работы)

К тебе приходит тимлид и говорит: Аналитики из команды продаж, просят предоставить им данные от внутреннего сервиса отвечающего за продажи определенных вещей. Они уже договорили с DBA(администратор баз данных) отвечающего за поддержание БД данного сервиса и он настроил топик в кролике(Rabbitmq, тоже самое что и кафка только в профиль), который находится в их структуре. Тебе необходимо положить эти данные в GreenPlum. Ты знаешь, что у вас в команде есть ClikHouse, Kafka, Spark, GreenPlum, но из Spark нельзя достучаться до кролика, так как он находится в другой сети. Spark может читать стримовые данные только из локальной кафки. Доступ есть у Клика, так как та самая комадна кладёт туда свои данные для анализа.

Твоя задача: взять данные из кролика(в роли кролика у нас выступает кафка с топиком **source.public.dbz_heartbeat**) и загнать данные в другой топик с именем **<фамилия>.source.data_from_rabit**. От туда твой мнимый товарисчь, мнимо данные заберёт и положит в мнимый GreenPlum

In [12]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.rabbit;
''')

client.command(f'''
    CREATE TABLE {database}.rabbit
    (
        json_kafka String 
    )
    ENGINE = Kafka
    SETTINGS
        kafka_broker_list = 'kafka:29093',
        kafka_topic_list = 'source.public.dbz_heartbeat', -- создай такой же топик
        kafka_group_name = 'clickhouse_consumer_group',
        kafka_format = 'JSONEachRow';
''')   


# Здесь создается таблица, которая обращается к source.public.dbz_heartbeat из "RabbitMQ"

In [13]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.from_rabbit;
''')

client.command(f'''
    CREATE TABLE {database}.from_rabbit
    (
      before String,
      after String                              -- задается только 1 строка с полным JSON-ом из кафки
    )
    ENGINE = Kafka
    SETTINGS
        kafka_broker_list = 'kafka:29093',
        kafka_topic_list = 'artemw9.source.data_from_rabbit',
        kafka_group_name = 'clickhouse_consumer_group',
        kafka_format = 'JSONAsString';                  -- достаточно часто истользуется именно этот формат
''')
# Здесь создается таблица-хранилище

In [15]:
client.command(f'''
    DROP VIEW IF EXISTS {database}.data_rabbit_mv;
''')

client.command(f'''
   CREATE MATERIALIZED VIEW {database}.data_rabbit_mv TO {database}.from_rabbit AS 
      SELECT 
         simpleJSONExtractRaw(json_kafka, 'before') as before,
         simpleJSONExtractRaw(json_kafka, 'after') as after
         -- для примера тут приведено всего 2 строки, но надеюсь принцип вы поняли
      from {database}.rabbit ;
''')
#Создается MV для записи в хранилище

### Решение(Смотреть только после выполенения)

In [9]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.data_from_rabbit;
''')

client.command(f'''
   create table {database}.data_from_rabbit 
      (
         json_kafka String
      )
   Engine = Kafka
   SETTINGS
        kafka_broker_list = 'kafka:29093',
        kafka_topic_list = 'source.public.dbz_heartbeat',
        kafka_group_name = 'clickhouse_consumer_group',
        kafka_format = 'JSONAsString'; 
''')

client.command(f'''
    DROP TABLE IF EXISTS {database}.data_rabbit_to_kafka;
''')

client.command(f'''
   create table {database}.data_rabbit_to_kafka 
   (
      before String,
      after String
   )
   Engine = Kafka
   SETTINGS
        kafka_broker_list = 'kafka:29093',
        kafka_topic_list = 'shustikov.source.data_from_rabit',
        kafka_group_name = 'clickhouse_consumer_group',
        kafka_format = 'JSONEachRow';
''')

client.command(f'''
    DROP VIEW IF EXISTS {database}.data_from_rabbit_mv;
''')

client.command(f'''
   CREATE MATERIALIZED VIEW {database}.data_from_rabbit_mv TO {database}.data_rabbit_to_kafka AS 
      SELECT 
         simpleJSONExtractRaw(json_kafka, 'before') as before,
         simpleJSONExtractRaw(json_kafka, 'after') as after
         -- для примера тут приведено всего 2 строки, но надеюсь принцип вы поняли
      from {database}.data_from_rabbit ;
''')

**Распределяем таблицу по шардам**

Если у вас будет распределенный ClickHouse, то данный сценарий у вас будет постояно

In [14]:
#query_df -- для вывода на экран

client.command(f'''
    CREATE TABLE {database}.events ON CLUSTER 'company_cluster' (   -- имя кластера можно задавать по имени и с помощью макроса '{{cluster}}'
        time DateTime,
        uid  Int64,
        type LowCardinality(String)
    )
    ENGINE = ReplicatedMergeTree('/clickhouse/tables/{{cluster}}/{{shard}}/events', '{{replica}}')
    PARTITION BY toDate(time)
    ORDER BY (uid);
''')

client.command(f'''
    CREATE TABLE {database}.events_distr ON CLUSTER 'company_cluster' AS {database}.events
    ENGINE = Distributed('company_cluster', {database}, events, uid);
''')

client.command(f'''
    INSERT INTO {database}.events_distr VALUES
        ('2020-01-01 10:00:00', 100, 'view'),
        ('2020-01-01 10:05:00', 101, 'view'),
        ('2020-01-01 11:00:00', 100, 'contact'),
        ('2020-01-01 12:10:00', 101, 'view'),
        ('2020-01-02 08:10:00', 100, 'view'),
        ('2020-01-03 13:00:00', 103, 'view');
''')


DatabaseError: Received ClickHouse exception, code: 60, server response: Code: 60. DB::Exception: Table artemw9.events_distr doesn't exist. (UNKNOWN_TABLE) (version 22.5.4.19 (official build)) (for url http://clickhouse01:8123)

In [10]:
client.query_df(f'''
    select * from {database}.events_distr 
''')

DatabaseError: Received ClickHouse exception, code: 60, server response: Code: 60. DB::Exception: Table artemw9.events_distr doesn't exist. (UNKNOWN_TABLE) (version 22.5.4.19 (official build)) (for url http://clickhouse01:8123)

In [15]:
client.query_df(f'''
    select * from {database}.events
''')

DatabaseError: Received ClickHouse exception, code: 60, server response: Code: 60. DB::Exception: Table artemw9.events doesn't exist. (UNKNOWN_TABLE) (version 22.5.4.19 (official build)) (for url http://clickhouse01:8123)

# **JOIN**

Ошибка распредленного джойна

In [4]:
client.command(f'''
    drop table {database}.tabl_join_local_1 on CLUSTER '{{cluster}}';
''')

client.command(f'''
    CREATE TABLE {database}.tabl_join_local_1 on CLUSTER '{{cluster}}'
    (
      id1 UInt32,
      id2 UInt32 
    )
    engine = MergeTree
    order by (id1);
''')

client.command(f'''
    drop table {database}.tabl_join_1 on CLUSTER '{{cluster}}';
''')

client.command(f'''
    CREATE TABLE {database}.tabl_join_1 ON CLUSTER '{{cluster}}' AS {database}.tabl_join_local_1
    ENGINE = Distributed('company_cluster', {database}, tabl_join_local_1, id1);
''')

client.command(f'''
    INSERT INTO {database}.tabl_join_1 values (1, 10),(2, 11),(3, 12),(4, 13),(5, 14),(6, 15),(7, 16),(8, 17),(9, 18),(0, 29)
''')

In [5]:
client.command(f'''
    drop table {database}.tabl_join_local_2 on CLUSTER '{{cluster}}';
''')

client.command(f'''
    CREATE TABLE {database}.tabl_join_local_2 on CLUSTER '{{cluster}}'
    (
      id1 UInt32,
      id2 UInt32 
    )
    engine = MergeTree
    order by (id1);
''')

client.command(f'''
    drop table {database}.tabl_join_2 on CLUSTER '{{cluster}}';
''')

client.command(f'''
    CREATE TABLE {database}.tabl_join_2 ON CLUSTER '{{cluster}}' AS {database}.tabl_join_local_2
    ENGINE = Distributed('company_cluster', {database}, tabl_join_local_2, id2);                                              -- изменен парамент распределения по шардам
''')

client.command(f'''
    INSERT INTO {database}.tabl_join_2 values (1, 10),(2, 11),(3, 12),(4, 13),(5, 14),(6, 15),(7, 16),(8, 17),(9, 19),(0, 29)
''')                                                                                                                          #(9, 19) попадет на один и тот же шард с (9, 18)

Посмотрим какие данные у нас на этой шарде, поменяй потом на tabl_join_local_2. Распределение данных по шардам основывается на остатке от деления

In [9]:

client.query_df(f'''
    select *
    from {database}.tabl_join_local_2
''')

,id1,id2
0,1,10
1,3,12
2,5,14
3,7,16


Попробуем сделать JOIN таблиц, но.....

In [19]:
# Получишь ошибку, так как кликхаус из коробки не зарешает распределённо джойнить
client.query_df(f'''
    select *
    from {database}.tabl_join_1 as t1  
      JOIN {database}.tabl_join_2 as t2
        ON t1.id1 = t2.id1
''')

DatabaseError: Received ClickHouse exception, code: 288, server response: Code: 288. DB::Exception: Double-distributed IN/JOIN subqueries is denied (distributed_product_mode = 'deny'). You may rewrite query to use local tables in subqueries, or use GLOBAL keyword, or set distributed_product_mode to suitable value.: While processing  artemw9.tabl_join_2 AS t2: While processing  INNER JOIN artemw9.tabl_join_2 AS t2 ON t1.id1 = t2.id1. (DISTRIBUTED_IN_JOIN_SUBQUERY_DENIED) (version 22.5.4.19 (official build)) (for url http://clickhouse01:8123)

GLOBAL - перекачивает данные из всех шардов на координатор и на данной тачке джойнит все на нём, естественно повышая нагрузку на этоот координатор

In [20]:
client.query_df(f'''
    select *
    from {database}.tabl_join_1 as t1  
      GLOBAL JOIN {database}.tabl_join_2 as t2
        ON t1.id1 = t2.id1
''')

,id1,id2,t2.id1,t2.id2
0,0,29,0,29
1,2,11,2,11
2,4,13,4,13
3,6,15,6,15
4,8,17,8,17
5,1,10,1,10
6,3,12,3,12
7,5,14,5,14
8,7,16,7,16
9,9,18,9,19


Настройка distributed_product_mode = 'local' разрешает джойнить таблицы локально на каждой шарде. Это может привести к коализиям если не джойнить таблицы по ключу распредления, как в следующем примере

In [21]:
client.command('''
    SET distributed_product_mode = 'local'  -- по умолчанию deny
''')

client.query_df(f'''
    select *
    from {database}.tabl_join_1 as t1  
      JOIN {database}.tabl_join_2 as t2
        ON t1.id1 = t2.id1
''')

,id1,id2,t2.id1,t2.id2
0,9,18,9,19


**ASOF JOIN** -- приближенное значение по условию неравентсва

In [22]:
client.query_df('''
  SELECT 
      number AS k, 
      toDateTime('2020-10-10 10:30:00') + number * 100 as ts, 
      number * 10 AS a
  FROM system.numbers LIMIT 5
''')



,k,ts,a
0,0,2020-10-10 10:30:00+03:00,0
1,1,2020-10-10 10:31:40+03:00,10
2,2,2020-10-10 10:33:20+03:00,20
3,3,2020-10-10 10:35:00+03:00,30
4,4,2020-10-10 10:36:40+03:00,40


In [23]:
client.query_df('''
      SELECT number AS k, 
          toDateTime('2020-10-10 10:00:00') + number * 100 + 3 as ts, 
          number * 100 AS b
      FROM system.numbers
      LIMIT 5
    UNION ALL
      SELECT number AS k, 
          toDateTime('2020-10-10 11:00:00') + number * 100 + 3 as ts, 
          number * 1000 AS b
      FROM system.numbers
      LIMIT 5
    UNION ALL
      SELECT number AS k,
          toDateTime('2020-10-10 12:00:00') + number * 100 + 3 as ts,
          number * 10000 AS b
      FROM system.numbers
      LIMIT 5
''')
 

,k,ts,b
0,0,2020-10-10 11:00:03+03:00,0
1,1,2020-10-10 11:01:43+03:00,1000
2,2,2020-10-10 11:03:23+03:00,2000
3,3,2020-10-10 11:05:03+03:00,3000
4,4,2020-10-10 11:06:43+03:00,4000
5,0,2020-10-10 10:00:03+03:00,0
6,1,2020-10-10 10:01:43+03:00,100
7,2,2020-10-10 10:03:23+03:00,200
8,3,2020-10-10 10:05:03+03:00,300
9,4,2020-10-10 10:06:43+03:00,400


In [24]:
client.query_df('''
    SELECT T_A.k, T_A.ts,  T_B.ts, T_A.a, T_B.b
    FROM
        (
            SELECT number AS k, 
            toDateTime('2020-10-10 10:30:00') + number * 100 as ts, 
            number * 10 AS a
            FROM system.numbers
            LIMIT 5
        ) T_A
    ASOF JOIN
        (
            SELECT number AS k, 
            toDateTime('2020-10-10 10:00:00') + number * 100 + 3 as ts, 
            number * 100 AS b
            FROM system.numbers
            LIMIT 5
            UNION ALL
            SELECT number AS k, 
            toDateTime('2020-10-10 11:00:00') + number * 100 + 3 as ts, 
            number * 1000 AS b
            FROM system.numbers
            LIMIT 5
            UNION ALL
            SELECT number AS k,
            toDateTime('2020-10-10 12:00:00') + number * 100 + 3 as ts,
            number * 10000 AS b
            FROM system.numbers
            LIMIT 5
        ) T_B ON
        T_A.k = T_B.k and
        T_A.ts < T_B.ts        
    ORDER BY T_A.k
        SETTINGS join_use_nulls = 0
''')




,k,ts,T_B.ts,a,b
0,0,2020-10-10 10:30:00+03:00,2020-10-10 11:00:03+03:00,0,0
1,1,2020-10-10 10:31:40+03:00,2020-10-10 11:01:43+03:00,10,1000
2,2,2020-10-10 10:33:20+03:00,2020-10-10 11:03:23+03:00,20,2000
3,3,2020-10-10 10:35:00+03:00,2020-10-10 11:05:03+03:00,30,3000
4,4,2020-10-10 10:36:40+03:00,2020-10-10 11:06:43+03:00,40,4000


### Решение(Смотреть только после выполенения)

Реши задачу с собесендования

Дано:
Представим, что у нас есть интернет магазин, из которого мы собираем данные.
1) данные по логинам пользователей
- данные летят в нашу таблицу в режиме реального времени, каждый раз, когда пользователь логинится на сайт
- таблица: user_login
- поля:
- user_id Uint64
- login_time DateTime
- объем данных большой

2.данные по акциям
- данные летят в нашу таблицу в режиме реального времени, каждый раз, когда создается новая акция
- таблица event
- поля:
- event_name String
- start_time DateTime
- end_time DateTime
- объем данных маленький

Нужно посчитать кол-во логинов пользователя во время действия акйций за выбранный промежуток времени

Удачи!